<a href="https://colab.research.google.com/github/meghanapeddaprolu/AI-Clinical-Assistant/blob/main/AI_Clinical_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Clinical Assistant

This project fine-tunes a pretrained LLM using LoRA to summarize clinical notes into concise medical summaries.

In [1]:
!pip install -q transformers datasets peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 21.7 MB/s eta 0:00:00


In [3]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

In [4]:
data = {
    "clinical_note": [
        "Patient has fever, cough, and fatigue for 3 days. No breathing difficulty. Prescribed antibiotics and advised rest.",
        "Patient reports headache and nausea since yesterday. No history of migraine. Advised hydration and observation.",
        "Patient has high blood pressure and dizziness. Currently taking antihypertensive medication. Follow-up recommended."
    ],
    "summary": [
        "Patient presents with 3-day fever, cough, and fatigue. Started on antibiotics and advised rest.",
        "Patient presents with headache and nausea since yesterday. Advised hydration and observation.",
        "Patient with hypertension presents with dizziness. Currently on antihypertensive medication; follow-up recommended."
    ]
}

dataset = Dataset.from_dict(data)

dataset

Dataset({
    features: ['clinical_note', 'summary'],
    num_rows: 3
})

In [5]:
print(dataset)

Dataset({
    features: ['clinical_note', 'summary'],
    num_rows: 3
})


In [6]:
split_dataset = dataset.train_test_split(test_size=0.33, seed=42)

train_dataset = split_dataset["train"]
test_dataset = split_dataset["test"]

print("Training examples:", len(train_dataset))
print("Testing examples:", len(test_dataset))

Training examples: 2
Testing examples: 1


In [7]:
print("TRAINING:")
print(train_dataset)

print("\nTESTING:")
print(test_dataset)

TRAINING:
Dataset({
    features: ['clinical_note', 'summary'],
    num_rows: 2
})

TESTING:
Dataset({
    features: ['clinical_note', 'summary'],
    num_rows: 1
})


In [8]:
print("TRAINING DATA:")
for row in train_dataset:
    print(row["clinical_note"])

print("\nTESTING DATA:")
for row in test_dataset:
    print(row["clinical_note"])

TRAINING DATA:
Patient reports headache and nausea since yesterday. No history of migraine. Advised hydration and observation.
Patient has fever, cough, and fatigue for 3 days. No breathing difficulty. Prescribed antibiotics and advised rest.

TESTING DATA:
Patient has high blood pressure and dizziness. Currently taking antihypertensive medication. Follow-up recommended.


In [9]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

print("Model loaded successfully!")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully!


In [11]:
text = "Patient has fever and cough."

tokens = tokenizer.tokenize(text)

print(tokens)

['Patient', 'Ġhas', 'Ġfever', 'Ġand', 'Ġcough', '.']


In [12]:
def tokenize_function(example):
    return tokenizer(
        example["clinical_note"],
        text_target=example["summary"],
        truncation=True
    )

tokenized_dataset = dataset.map(tokenize_function)

print(tokenized_dataset)

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Dataset({
    features: ['clinical_note', 'summary', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 3
})


In [13]:
print(tokenized_dataset[0])

{'clinical_note': 'Patient has fever, cough, and fatigue for 3 days. No breathing difficulty. Prescribed antibiotics and advised rest.', 'summary': 'Patient presents with 3-day fever, cough, and fatigue. Started on antibiotics and advised rest.', 'input_ids': [36592, 702, 33553, 11, 39600, 11, 323, 35609, 369, 220, 18, 2849, 13, 2308, 25938, 16829, 13, 4111, 17433, 45750, 323, 25104, 2732, 13], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [36592, 18404, 448, 220, 18, 11228, 33553, 11, 39600, 11, 323, 35609, 13, 35812, 389, 45750, 323, 25104, 2732, 13]}


In [29]:
!pip install -U "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 54.1 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [31]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:305: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093


In [15]:
import torch
import peft
import transformers

print("PyTorch:", torch.__version__)
print("PEFT:", peft.__version__)
print("Transformers:", transformers.__version__)

PyTorch: 2.11.0+cpu
PEFT: 0.20.0
Transformers: 5.15.0


In [32]:
model = AutoModelForCausalLM.from_pretrained(model_name)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093


In [33]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./clinical_lora_model",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="no",
    report_to="none"
)

In [34]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

In [38]:
def format_example(example):
    text = (
        "Summarize the following clinical note:\n"
        + example["clinical_note"]
        + "\nSummary:\n"
        + example["summary"]
    )
    return {"text": text}

formatted_train = train_dataset.map(format_example)
formatted_test = test_dataset.map(format_example)

print(formatted_train[0])

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

{'clinical_note': 'Patient reports headache and nausea since yesterday. No history of migraine. Advised hydration and observation.', 'summary': 'Patient presents with headache and nausea since yesterday. Advised hydration and observation.', 'text': 'Summarize the following clinical note:\nPatient reports headache and nausea since yesterday. No history of migraine. Advised hydration and observation.\nSummary:\nPatient presents with headache and nausea since yesterday. Advised hydration and observation.'}


In [41]:
def tokenize_formatted(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=512
    )

tokenized_train = formatted_train.map(tokenize_formatted)
tokenized_test = formatted_test.map(tokenize_formatted)

print(tokenized_train[0])

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

{'clinical_note': 'Patient reports headache and nausea since yesterday. No history of migraine. Advised hydration and observation.', 'summary': 'Patient presents with headache and nausea since yesterday. Advised hydration and observation.', 'text': 'Summarize the following clinical note:\nPatient reports headache and nausea since yesterday. No history of migraine. Advised hydration and observation.\nSummary:\nPatient presents with headache and nausea since yesterday. Advised hydration and observation.', 'input_ids': [9190, 5612, 551, 279, 2701, 14490, 5185, 510, 36592, 6682, 46746, 323, 60780, 2474, 13671, 13, 2308, 3840, 315, 91881, 13, 23924, 291, 86900, 323, 21930, 624, 19237, 510, 36592, 18404, 448, 46746, 323, 60780, 2474, 13671, 13, 23924, 291, 86900, 323, 21930, 13], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [42]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

print("Data collator created successfully!")

Data collator created successfully!


In [43]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    data_collator=data_collator,
)

print("Trainer is ready!")

Trainer is ready!


In [44]:
trainer.train()

Step,Training Loss
1,2.624812
2,2.432380
3,2.483853
4,2.329378
5,2.376192
6,2.227140


TrainOutput(global_step=6, training_loss=2.4122923215230307, metrics={'train_runtime': 397.8755, 'train_samples_per_second': 0.015, 'train_steps_per_second': 0.015, 'total_flos': 632286014976.0, 'train_loss': 2.4122923215230307, 'epoch': 3.0})

In [45]:
test_note = test_dataset[0]["clinical_note"]

print("Unseen clinical note:")
print(test_note)

Unseen clinical note:
Patient has high blood pressure and dizziness. Currently taking antihypertensive medication. Follow-up recommended.


In [46]:
prompt = "Summarize the following clinical note:\n" + test_note + "\nSummary:\n"

inputs = tokenizer(prompt, return_tensors="pt")

outputs = model.generate(
    **inputs,
    max_new_tokens=50
)

generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(generated_text)

Summarize the following clinical note:
Patient has high blood pressure and dizziness. Currently taking antihypertensive medication. Follow-up recommended.
Summary:
The patient has been diagnosed with hypertension and is currently receiving antihypertensive medication. A follow-up appointment is scheduled for a later date. 

Explanation: 
- "High blood pressure" refers to an elevated level of sodium in the blood,


In [47]:
expected_summary = test_dataset[0]["summary"]

print("MODEL OUTPUT:")
print(generated_text)

print("\nEXPECTED SUMMARY:")
print(expected_summary)

MODEL OUTPUT:
Summarize the following clinical note:
Patient has high blood pressure and dizziness. Currently taking antihypertensive medication. Follow-up recommended.
Summary:
The patient has been diagnosed with hypertension and is currently receiving antihypertensive medication. A follow-up appointment is scheduled for a later date. 

Explanation: 
- "High blood pressure" refers to an elevated level of sodium in the blood,

EXPECTED SUMMARY:
Patient with hypertension presents with dizziness. Currently on antihypertensive medication; follow-up recommended.


In [48]:
model.save_pretrained("./clinical_lora_model")
tokenizer.save_pretrained("./clinical_lora_model")

print("Fine-tuned model saved successfully!")

Fine-tuned model saved successfully!


In [49]:
import os

print(os.listdir("./clinical_lora_model"))

['tokenizer_config.json', 'adapter_config.json', 'tokenizer.json', 'chat_template.jinja', 'README.md', 'adapter_model.safetensors']


In [50]:
from difflib import SequenceMatcher

score = SequenceMatcher(
    None,
    generated_text.lower(),
    expected_summary.lower()
).ratio()

print("Similarity Score:", round(score, 3))

Similarity Score: 0.355


In [51]:
!pip install -q -U google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.1/259.1 kB 19.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.56.3 which is incompatible.


In [52]:
from google.colab import userdata
from google import genai

client = genai.Client(
    api_key=userdata.get("GEMINI_API_KEY")
)

print("Gemini client ready!")

Gemini client ready!


In [53]:
prompt = """
Explain the trade-offs between these four LLM fine-tuning methods:

1. Full Fine-Tuning
2. LoRA
3. QLoRA
4. DoRA

Compare them based on:
- trainable parameters
- GPU memory requirements
- training speed
- performance
- suitability for healthcare clinical-note summarization

Explain it in beginner-friendly language.
"""

response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents=prompt
)

print(response.text)

Fine-tuning a Large Language Model (LLM) is like taking a smart college graduate (a base model) and sending them to a specialized school to become a specialist (like a medical doctor). 

Different fine-tuning methods are like different study strategies. Some require buying a whole library, while others use clever shortcuts to save time and money.

Here is a beginner-friendly breakdown of **Full Fine-Tuning (FFT)**, **LoRA**, **QLoRA**, and **DoRA**, compared across your criteria.

---

### The Analogy: Understanding the 4 Methods

*   **1. Full Fine-Tuning (FFT) – "The Brain Transplant":** You rewrite and update every single neuron in the LLM's brain. It’s highly effective but requires a massive amount of energy and space.
*   **2. LoRA (Low-Rank Adaptation) – "The Side-Notes":** You freeze the LLM's brain. Instead of changing the original knowledge, you attach a small notebook to the side. The LLM reads its original brain *plus* your side-notes to make decisions.
*   **3. QLoRA (Quant

In [54]:
gemini_explanation = response.text

print("GEMINI EXPLANATION")
print("=" * 50)
print(gemini_explanation)

GEMINI EXPLANATION
Fine-tuning a Large Language Model (LLM) is like taking a smart college graduate (a base model) and sending them to a specialized school to become a specialist (like a medical doctor). 

Different fine-tuning methods are like different study strategies. Some require buying a whole library, while others use clever shortcuts to save time and money.

Here is a beginner-friendly breakdown of **Full Fine-Tuning (FFT)**, **LoRA**, **QLoRA**, and **DoRA**, compared across your criteria.

---

### The Analogy: Understanding the 4 Methods

*   **1. Full Fine-Tuning (FFT) – "The Brain Transplant":** You rewrite and update every single neuron in the LLM's brain. It’s highly effective but requires a massive amount of energy and space.
*   **2. LoRA (Low-Rank Adaptation) – "The Side-Notes":** You freeze the LLM's brain. Instead of changing the original knowledge, you attach a small notebook to the side. The LLM reads its original brain *plus* your side-notes to make decisions.
* 

In [55]:
print("AI CLINICAL ASSISTANT - FINE-TUNING RESULTS")
print("=" * 50)

print("Base Model: Qwen/Qwen2.5-0.5B-Instruct")
print("Fine-Tuning Method: LoRA")
print("Trainable Parameters: 540,672")
print("Trainable Percentage: 0.109%")
print("Training Epochs: 3")
print("Training Examples: 2")
print("Testing Examples: 1")
print("Model: Fine-tuned successfully")
print("Evaluation: Completed")
print("Gemini Interpretation: Completed")

AI CLINICAL ASSISTANT - FINE-TUNING RESULTS
Base Model: Qwen/Qwen2.5-0.5B-Instruct
Fine-Tuning Method: LoRA
Trainable Parameters: 540,672
Trainable Percentage: 0.109%
Training Epochs: 3
Training Examples: 2
Testing Examples: 1
Model: Fine-tuned successfully
Evaluation: Completed
Gemini Interpretation: Completed
